In [ ]:
%pip install --upgrade google-cloud-vectorsearch fsspec gcsfs google-auth google-api-core

In [4]:
import google.auth
_, PROJECT_ID = google.auth.default()
LOCATION="asia-southeast1"
COLLECTION_ID="amazon-product-dataset-768-compact"

In [2]:
import pandas as pd
df_golden_product = pd.read_parquet('gs://jk-amazon-products-index/golden-records/amazon_products_golden.parquet')
df_golden_product

,id,vectors,product_type,item_name,item_keywords,item_description,data
0,00000529,"{'image_embedding': [-0.03478194, -0.005159070...",HOME,[DE] UMI Slim Laundry Basket on Wheels Tall Ro...,,[DE] UMI Slim Laundry Basket on Wheels Tall Ro...,{'description': '[DE] UMI Slim Laundry Basket ...
1,00003a93,"{'image_embedding': [-0.028530316, -0.03578147...",HOME,[ES] UMI. Essentials - Marco Doble para Sobrem...,#Marcos de Foto de estar dormitorio oficina cu...,[ES] UMI. Essentials - Marco Doble para Sobrem...,{'description': '[ES] UMI. Essentials - Marco ...
2,000088e1,"{'image_embedding': [-0.03092741, 0.0020457155...",LAUNDRY_HAMPER,[ES] Eono Amazon Brand Doble Bolsas Cestos par...,#Gran cesta de lavanderia lavado de ropa plega...,[ES] Eono Amazon Brand Doble Bolsas Cestos par...,{'description': '[ES] Eono Amazon Brand Doble ...
3,0000b301,"{'image_embedding': [0.009060884, -0.030949578...",HOME_FURNITURE_AND_DECOR,[MX] Stone & Beam Alfombra Moderna a Rayas,,[MX] Stone & Beam Alfombra Moderna a Rayas\n\n,{'description': '[MX] Stone & Beam Alfombra Mo...
4,0000b9b8,"{'image_embedding': [-0.044396702, -0.01553370...",WRENCH,[ES] AmazonBasics - Juego de llaves de trinque...,,[ES] AmazonBasics - Juego de llaves de trinque...,{'description': '[ES] AmazonBasics - Juego de ...
...,...,...,...,...,...,...,...
398184,ffff4040,"{'image_embedding': [-0.011574976, 0.01045859,...",CHAIR,[CA] AmazonBasics Multi-Purpose Adjustable Off...,#small #makeup #de #big #armless #medical #w/m...,[CA] AmazonBasics Multi-Purpose Adjustable Off...,{'description': '[CA] AmazonBasics Multi-Purpo...
398185,ffff87d7,"{'image_embedding': [-0.047705553, -0.05032951...",HUMIDIFIER,[IN] AmazonBasics Humidifier with Smart Auto-H...,#small #rooms #4010 #homech #humifiers #vicks ...,[IN] AmazonBasics Humidifier with Smart Auto-H...,{'description': '[IN] AmazonBasics Humidifier ...
398186,ffffc378,"{'image_embedding': [0.011752412, -0.017997367...",GROCERY,"[US] 365 Everyday Value, Raspberry Geranium Fr...",#sorbet #dessert #botanical #floral #flowers #...,"[US] 365 Everyday Value, Raspberry Geranium Fr...","{'description': '[US] 365 Everyday Value, Rasp..."
398187,ffffd5c6,"{'image_embedding': [-0.030390358, 0.003537298...",STORAGE_BAG,[ES] AmazonBasics 3-Bag Laundry Hamper Sorter ...,,[ES] AmazonBasics 3-Bag Laundry Hamper Sorter ...,{'description': '[ES] AmazonBasics 3-Bag Laund...


In [9]:
set(df_golden_product['product_type'].tolist())

{'ABIS_BEAUTY',
 'ABIS_BOOK',
 'ABIS_DRUGSTORE',
 'ABIS_ELECTRONICS',
 'ABIS_HOME_IMPROVEMENT',
 'ABIS_KITCHEN',
 'ABIS_LAWN_AND_GARDEN',
 'ABIS_PET_PRODUCTS',
 'ABIS_VIDEO_GAMES',
 'ACCESSORY',
 'ACCESSORY_OR_PART_OR_SUPPLY',
 'AGRICULTURAL_SUPPLIES',
 'AIR_COMPRESSOR',
 'AIR_CONDITIONER',
 'AIR_FRYER',
 'AIR_MATTRESS',
 'AIR_PUMP',
 'AIR_PURIFIER',
 'AMAZON_BOOK_READER_ACCESSORY',
 'AMAZON_TABLET_ACCESSORY',
 'ANIMAL_COLLAR',
 'ANIMAL_LITTER',
 'ANTENNA',
 'AREA_DEODORIZER',
 'ARTIFICIAL_PLANT',
 'ARTIFICIAL_TREE',
 'ART_AND_CRAFT_SUPPLY',
 'ASTRINGENT_SUBSTANCE',
 'AUDIO_OR_VIDEO',
 'AUTO_ACCESSORY',
 'AUTO_CHEMICAL',
 'AUTO_OIL',
 'AUTO_PART',
 'AV_FURNITURE',
 'AV_RECEIVER',
 'BABY_BOTTLE',
 'BABY_PRODUCT',
 'BACKPACK',
 'BADGE_HOLDER',
 'BAG',
 'BAKEWARE',
 'BAKING_CUP',
 'BAKING_MIX',
 'BAKING_PAN',
 'BAKING_PAPER',
 'BARBECUE_GRILL',
 'BARBELL',
 'BASKET',
 'BATHWATER_ADDITIVE',
 'BATTERY',
 'BEAN_BAG_CHAIR',
 'BEAUTY',
 'BED',
 'BED_FRAME',
 'BENCH',
 'BINOCULAR',
 'BISS',
 'B

In [3]:
product_filter = [
    # 가전 / 디지털 / IT
    'CELLULAR_PHONE', 'NOTEBOOK_COMPUTER', 
    'PERSONAL_COMPUTER', 'MONITOR', 'KEYBOARDS', 'INPUT_MOUSE', 
    'HEADPHONES', 'SPEAKERS', 'COMPUTER_SPEAKER', 'MICROPHONE', 'TELEVISION', 
    'REMOTE_CONTROL', 'BATTERY', 'POWER_BANK', 'CHARGING_ADAPTER', 'FLASH_DRIVE', 
    'WEIGH_SCALE', 'CLOCK', 'SECURITY_CAMERA',
    
    # 주방 / 식기    
    'KITCHEN_KNIFE', 'SCISSORS', 'CAN_OPENER', 'BOTTLE_OPENER', 
    'DINNERWARE', 'DISHWARE_PLATE', 'DISHWARE_BOWL', 'DRINKING_CUP', 
    'FLATWARE', 'DRINKING_STRAW', 'DRINK_COASTER', 'PITCHER', 'THERMOS', 
    'PLACEMAT', 'POT_HOLDER', 'BOTTLE_RACK', 'DRYING_RACK', 'FOOD_STORAGE_BAG',

    # 뷰티 / 생활용품 / 위생
    'HAIR_BRUSH', 'HAIR_COMB', 'HAIR_IRON', 
    'SKIN_CLEANING_AGENT', 'SKIN_MOISTURIZER', 'SKIN_EXFOLIANT', 'SKIN_TREATMENT_MASK', 
    'SUNSCREEN', 'LIP_BALM', 'TOOTHBRUSH', 'TOOTH_CLEANING_AGENT', 
    'MOUTHWASH', 'FACIAL_TISSUE', 'SKIN_CLEANING_WIPE', 'CLEANING_AGENT', 
    'CLEANING_BRUSH', 'BROOM', 'MASCARA', 'NAIL_POLISH',

    # 식품 / 음료 / 식재료
    'WATER', 'COFFEE', 'TEA', 'JUICE_AND_JUICE_DRINK', 'MILK_SUBSTITUTE', 
    'BREAD', 'COOKIE', 'CAKE', 'PASTRY', 'CRACKER', 'PRETZEL', 
    'POPCORN', 'CANDY', 'CHOCOLATE_CANDY', 'SUGAR_CANDY', 'SNACK_CHIP_AND_CRISP', 
    'SNACK_FOOD_BAR', 'BREAKFAST_CEREAL', 'NOODLE', 'RICE_MIX', 'PACKAGED_SOUP_AND_STEW', 
    'MEAT', 'POULTRY', 'FISH', 'SHELLFISH', 'VEGETABLE', 'FRUIT', 
    'NUTS', 'LEGUME', 'FLOUR', 'SUGAR', 'SUGAR_SUBSTITUTE', 'HONEY', 
    'EDIBLE_OIL_VEGETABLE', 'DAIRY_BASED_BUTTER', 'DAIRY_BASED_CHEESE', 'DAIRY_BASED_CREAM', 
    'SAUCE', 'SALAD_DRESSING', 'VINEGAR', 'WINE',

    # 가구 / 인테리어 / 수납
    'BLANKET', 'SOFA', 'CHAIR', 'DESK', 'TABLE', 'BENCH',  
    'CABINET', 'SHELF', 'CURTAIN', 'HOME_MIRROR', 
    'VASE', 'TRASH_CAN', 'STORAGE_BOX', 'STORAGE_BAG',

    # 패션잡화 / 취미 / 반려용품
    'SHOES', 'SANDAL', 'BOOT', 'BAG', 'BACKPACK', 
    'HANDBAG', 'TOTE_BAG', 'WALLET', 'WATCH', 'SUNGLASSES', 'EARRING', 'NECKLACE', 'BRACELET', 
    'RING', 

    # 문구 / 공구 / 건강 / 기타
    'WRITING_INSTRUMENT', 'STAPLER', 'SELF_STICK_NOTE', 'LOCK', 'VITAMIN', 
    'DIETARY_SUPPLEMENTS', 'FIRST_AID_KIT']
df_compact_products = df_golden_product[df_golden_product['product_type'].isin(product_filter)]
df_compact_products

,id,vectors,product_type,item_name,item_keywords,item_description,data
11,0001c52d,"{'image_embedding': [-0.019685043, -0.00303021...",CHAIR,[US] Stone & Beam S&B GF GF-80933G Vintage Sil...,#big #heavyweight #no #basics #espo #allsteel ...,[US] Stone & Beam S&B GF GF-80933G Vintage Sil...,{'description': '[US] Stone & Beam S&B GF GF-8...
12,00020b5a,"{'image_embedding': [-0.041058376, -0.03353369...",SHOES,[AU] find. Women's Kitten Heel Court Closed-To...,"#heel,kitten,pump,woman,women #UK #UK #UK #UK ...",[AU] find. Women's Kitten Heel Court Closed-To...,{'description': '[AU] find. Women's Kitten Hee...
13,00020b94,"{'image_embedding': [-0.019196399, 0.020137342...",EARRING,[JP] [アマゾンコレクション] Amazon Collection 14Kイエローゴール...,#イヤリングスタッドフープゴールドハローフック真珠ジュエリーブラック小さなヴィンテージオパー...,[JP] [アマゾンコレクション] Amazon Collection 14Kイエローゴール...,{'description': '[JP] [アマゾンコレクション] Amazon Coll...
19,00033168,"{'image_embedding': [-0.004150736, 0.018715905...",CHAIR,[MX] Strathwood Basics - Silla acolchada de po...,#Jugar #Grande #Tienda #Joe #camping #Rosa #Pl...,[MX] Strathwood Basics - Silla acolchada de po...,{'description': '[MX] Strathwood Basics - Sill...
20,0003658b,"{'image_embedding': [0.017919777, 0.00344414, ...",CRACKER,"[US] 365 Everyday Value, Oyster Crackers, 8 oz",,"[US] 365 Everyday Value, Oyster Crackers, 8 oz...","{'description': '[US] 365 Everyday Value, Oyst..."
...,...,...,...,...,...,...,...
398166,fffb48d7,"{'image_embedding': [0.001306995, 0.019842863,...",SHOES,[IN] Bourge Men's Magic-20 Navy Sneakers-11 UK...,#footwear #casual #gents #daily #casual #daily...,[IN] Bourge Men's Magic-20 Navy Sneakers-11 UK...,{'description': '[IN] Bourge Men's Magic-20 Na...
398180,fffe67d0,"{'image_embedding': [-0.057308294, 0.02074126,...",SHOES,[IN] Amazon Brand - Symbol Women's Tan Fashion...,#bellies for women stylish #shoes for women st...,[IN] Amazon Brand - Symbol Women's Tan Fashion...,{'description': '[IN] Amazon Brand - Symbol Wo...
398184,ffff4040,"{'image_embedding': [-0.011574976, 0.01045859,...",CHAIR,[CA] AmazonBasics Multi-Purpose Adjustable Off...,#small #makeup #de #big #armless #medical #w/m...,[CA] AmazonBasics Multi-Purpose Adjustable Off...,{'description': '[CA] AmazonBasics Multi-Purpo...
398187,ffffd5c6,"{'image_embedding': [-0.030390358, 0.003537298...",STORAGE_BAG,[ES] AmazonBasics 3-Bag Laundry Hamper Sorter ...,,[ES] AmazonBasics 3-Bag Laundry Hamper Sorter ...,{'description': '[ES] AmazonBasics 3-Bag Laund...


In [6]:
!gcloud storage buckets create gs://$PROJECT_ID-vs2 --location=asia-southeast1

Creating gs://sandbox-373102-vs2/...


In [7]:
# 원하는 컬럼('id', 'vectors')만 선택하여 JSONL로 저장
df_compact_products[['id', 'data', 'vectors']].to_json(f'gs://{PROJECT_ID}-vs2/data/amazon-product-dataset-768-compact.jsonl', orient='records', lines=True, force_ascii=False)

In [ ]:
from google.cloud import vectorsearch_v1beta

# Create the client
vector_search_service_client = vectorsearch_v1beta.VectorSearchServiceClient()

# The JSON schema for the data
data_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "description": {"type": "string"},
    },
}

# The JSON schema for the vector
vector_schema = {
    "image_embedding": {"dense_vector": {"dimensions": 768}},
    "text_embedding": {"dense_vector": {"dimensions": 768}}
}

collection = vectorsearch_v1beta.Collection(
    data_schema=data_schema,
    vector_schema=vector_schema,
)
request = vectorsearch_v1beta.CreateCollectionRequest(
    parent=f"projects/{PROJECT_ID}/locations/{LOCATION}",
    collection_id=COLLECTION_ID,
    collection=collection,
)

# Create the collection
operation = vector_search_service_client.create_collection(request=request)

# Wait for the result (note this may take up to several minutes)
operation.result()

In [ ]:
from datetime import datetime
from google.cloud import vectorsearch_v1

# Create the client
vector_search_service_client = vectorsearch_v1.VectorSearchServiceClient()

# Initialize request
request = vectorsearch_v1.ImportDataObjectsRequest(
    name=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}",
    gcs_import={
      "contents_uri": f"gs://{PROJECT_ID}-vs2/data/",
      "error_uri": f"gs://{PROJECT_ID}-vs2/error/",
    },
)

# Make the request
print(datetime.now()) 
operation = vector_search_service_client.import_data_objects(request=request)

In [ ]:
import time
while operation.done() == False:
    time.sleep(1)
print(datetime.now())

In [ ]:
from google.cloud import vectorsearch_v1beta

# Create a client
client = vectorsearch_v1beta.VectorSearchServiceClient()

operations = []
# Initialize request argument(s)
for index_field in ["image_embedding", "text_embedding"]:
    index = vectorsearch_v1beta.Index(
        index_field=index_field,
        #filter_fields=["year", "genre"],
        store_fields=["name", "description"],
    )
    request = vectorsearch_v1beta.CreateIndexRequest(
        parent=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}",
        index_id=f"idx-{index_field.replace('_', '-')}-amazon-product-dataset-768-compact",
        index=index,
    )
    
    # Make the request
    operations.append(client.create_index(request=request))
print(datetime.now())

In [ ]:
while not all(operation.done() for operation in operations):
    time.sleep(1)    
print(datetime.now())